<a href="https://colab.research.google.com/github/verrelshafryhermawan-boop/Tugas-AI/blob/main/PredikTugasANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# --- STEP 1: Load & Preprocess Dataset ---
# Memuat data dari file CSV yang diunggah
df = pd.read_csv('dataset rel.csv')

# Menghapus kolom 'No' karena tidak relevan untuk prediksi
df = df.drop(columns=['No'])

# Mengubah data teks (kategorikal) menjadi angka menggunakan LabelEncoder
# Contoh: 'Ya' -> 1, 'Tidak' -> 0
le = LabelEncoder()
for col in df.columns:
    df[col] = le.fit_transform(df[col])

# Memisahkan Fitur (X) dan Target/Label (y)
X_data = df.drop(columns=['Menang']).values
y_data = df['Menang'].values

# Normalisasi data agar perhitungan bobot lebih stabil
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_data)

# Reshape data agar sesuai dengan format fungsi ANN (fitur sebagai baris)
# X shape: (jumlah_fitur, jumlah_sampel)
# Y shape: (1, jumlah_sampel)
X_final = X_scaled.T
y_final = y_data.reshape(1, -1)

# --- STEP 2: Initializing the Neural Network ---
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }
    return parameters

# --- STEP 3: Defining Activation Functions ---
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

# --- STEP 4: Forward Propagation ---
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# --- STEP 5: Computing the Cost ---
def compute_cost(Y, A2):
    m = Y.shape[1]
    # Menambahkan epsilon kecil untuk menghindari log(0)
    epsilon = 1e-15
    cost = -np.sum(Y * np.log(A2 + epsilon) + (1 - Y) * np.log(1 - A2 + epsilon)) / m
    return np.squeeze(cost)

# --- STEP 6: Backpropagation ---
def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

# --- STEP 7: Updating Parameters ---
def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

# --- STEP 8: Training the Neural Network ---
def train_neural_network(X, Y, input_size, hidden_size, output_size, epochs=1000, learning_rate=0.01):
    parameters = initialize_parameters(input_size, hidden_size, output_size)

    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 500 == 0:
            print(f"Epoch {i}: Cost = {cost:.6f}")

    return parameters

# --- STEP 9: Making Predictions ---
def predict(X, parameters):
    A2, _ = forward_propagation(X, parameters)
    return (A2 > 0.5).astype(int)

# --- STEP 10: Running the Model ---
input_dim = X_final.shape[0]  # Jumlah fitur (5 kolom)
hidden_dim = 8                # Jumlah neuron di hidden layer
output_dim = 1                # Output biner (Menang/Tidak)

print("--- Memulai Pelatihan Model pada Dataset Rel ---")
trained_parameters = train_neural_network(
    X_final, y_final,
    input_size=input_dim,
    hidden_size=hidden_dim,
    output_size=output_dim,
    epochs=5000,
    learning_rate=0.05
)

# Menguji hasil prediksi pada data yang sama
predictions = predict(X_final, trained_parameters)
accuracy = np.mean(predictions == y_final) * 100

print(f"\n--- Hasil Akhir ---")
print(f"Akurasi Pelatihan: {accuracy:.2f}%")
print("Contoh 5 Prediksi Pertama:", predictions[0][:5])
print("Contoh 5 Target Asli    :", y_final[0][:5])

--- Memulai Pelatihan Model pada Dataset Rel ---
Epoch 0: Cost = 0.693151
Epoch 500: Cost = 0.114987
Epoch 1000: Cost = 0.055806
Epoch 1500: Cost = 0.036595
Epoch 2000: Cost = 0.025745
Epoch 2500: Cost = 0.019188
Epoch 3000: Cost = 0.014993
Epoch 3500: Cost = 0.012160
Epoch 4000: Cost = 0.010147
Epoch 4500: Cost = 0.008662

--- Hasil Akhir ---
Akurasi Pelatihan: 100.00%
Contoh 5 Prediksi Pertama: [1 0 0 1 0]
Contoh 5 Target Asli    : [1 0 0 1 0]
